# OpenGraph Image — Quickstart: From Image to Knowledge Graph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenGraphAI/opengraph-ai/blob/main/opengraph-image/notebooks/opengraph_image_quickstart.ipynb)

In this notebook you will:
- Install the `opengraph-image` package
- Load a sample image
- Extract a knowledge graph with Claude vision
- Save the extraction to `graph.json`
- Render an interactive D3.js `graph.html`
- (Optional) Query the graph with natural language

> **Requires an `ANTHROPIC_API_KEY`.** Get one from the [Anthropic Console](https://console.anthropic.com/) and add it as a Colab secret (or you'll be prompted to paste it below).

## What is opengraph-image?

`opengraph-image` turns a single image into a structured knowledge graph: objects, the overall scene, visual attributes (color, material, mood, lighting, style, size), and any text found in the image, all connected by typed edges. It uses Claude's vision capability to do the extraction in a single call, validates the result against a strict Pydantic schema, and stores everything in a local NetworkX graph that you can save, merge across images, or query in natural language. No separate object-detection or OCR model is required — one multimodal call does it all. This quickstart uses Claude vision. A fine-tuned Gemma 4 E4B path is planned — see the fine-tuning guidebook in `opengraph-samples`.

## Setup development environment

Install `opengraph-image` and its dependencies directly from GitHub, then configure your Anthropic API key.

In [1]:
!pip install -q git+https://github.com/OpenGraphAI/opengraph-ai.git#subdirectory=opengraph-image
!pip install -q anthropic pillow networkx python-dotenv ipython

ERROR: Package 'opengraph-image' requires a different Python: 3.10.16 not in '>=3.11'


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

## Load a test image

To keep this notebook self-contained on Colab (no local filesystem, no manual uploads), we pull a real sample image straight from the `opengraph-image` repo's `tests/sample_images/test_photos/` folder.

In [ ]:
!wget -q -O sample.jpg https://raw.githubusercontent.com/OpenGraphAI/opengraph-ai/main/opengraph-image/tests/sample_images/test_photos/park-2.jpg

from IPython.display import Image
Image("sample.jpg", width=480)

## Extract the knowledge graph

`extract_image` sends the image to Claude vision and returns an `ImageExtraction`: an `ImageNode` for the image itself, `ObjectNode`s for detected objects, a `SceneNode` for the overall context, `AttributeNode`s for visual qualities like color and mood, and `TextSpanNode`s for any visible text — all wired together with five edge types (`contains`, `in_scene`, `has_attribute`, `has_text`, `related_to`).

In [ ]:
from pathlib import Path
import anthropic
from opengraph_image.extract import extract_image

client = anthropic.Anthropic()
extraction = extract_image(Path("sample.jpg"), client)

print(f"Objects: {len(extraction.objects)}")
print(f"Scene:   {extraction.scenes[0].label if extraction.scenes else '—'}")
print(f"Sample object labels: {[o.label for o in extraction.objects[:5]]}")

## Build graph.json

In [ ]:
import json
from opengraph_image.graph import ImageGraph

graph = ImageGraph()
graph.add_extraction(extraction)
graph.to_json(Path("graph.json"))
print(json.dumps(graph.summary(), indent=2))

## Visualize with D3.js

`render_graph_html_d3` (defined below) reads `graph.json` and writes a self-contained, interactive HTML page: nodes are colored by type (image / object / scene / attribute / text_span), laid out with a D3 force simulation, and support dragging, zooming, and hover tooltips showing each node's id and label.

In [ ]:
import json
from pathlib import Path

_HTML_TEMPLATE = """<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body { margin: 0; padding: 0; background: transparent; }
  svg { width: 100%; height: 100%; background: transparent; }
  .link { stroke: #94a3b8; stroke-opacity: 0.6; stroke-width: 1.5px; }
  .node circle { stroke: #fff; stroke-width: 1.5px; cursor: grab; }
  .node text { font: 10px sans-serif; fill: #334155; pointer-events: none; }
</style>
</head>
<body>
<svg id="graph" width="900" height="600" viewBox="0 0 900 600"></svg>
<script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
<script>
const graphData = __GRAPH_DATA__;

const width = 900;
const height = 600;

const color = {
  image: "#2563eb",
  object: "#f97316",
  scene: "#10b981",
  attribute: "#8b5cf6",
  text_span: "#6b7280"
};

const svg = d3.select("#graph");
const container = svg.append("g");

svg.call(
  d3.zoom()
    .scaleExtent([0.2, 8])
    .on("zoom", (event) => container.attr("transform", event.transform))
);

svg.append("defs").append("marker")
  .attr("id", "arrow")
  .attr("viewBox", "0 -5 10 10")
  .attr("refX", 18)
  .attr("refY", 0)
  .attr("markerWidth", 6)
  .attr("markerHeight", 6)
  .attr("orient", "auto")
  .append("path")
  .attr("d", "M0,-5L10,0L0,5")
  .attr("fill", "#cbd5e1");

const simulation = d3.forceSimulation(graphData.nodes)
  .force("link", d3.forceLink(graphData.links).id(d => d.id).distance(70))
  .force("charge", d3.forceManyBody().strength(-220))
  .force("center", d3.forceCenter(width / 2, height / 2));

const link = container.append("g")
  .selectAll("line")
  .data(graphData.links)
  .join("line")
  .attr("class", "link")
  .attr("marker-end", "url(#arrow)");

const node = container.append("g")
  .selectAll("g")
  .data(graphData.nodes)
  .join("g")
  .attr("class", "node")
  .call(drag(simulation));

node.append("circle")
  .attr("r", d => (d.type === "image" ? 10 : 6))
  .attr("fill", d => color[d.type] || "#9ca3af");

node.append("title")
  .text(d => `${d.id} — ${d.label}`);

node.append("text")
  .attr("dx", 10)
  .attr("dy", 4)
  .text(d => d.label);

simulation.on("tick", () => {
  link
    .attr("x1", d => d.source.x)
    .attr("y1", d => d.source.y)
    .attr("x2", d => d.target.x)
    .attr("y2", d => d.target.y);

  node.attr("transform", d => `translate(${d.x},${d.y})`);
});

function drag(sim) {
  function dragstarted(event, d) {
    if (!event.active) sim.alphaTarget(0.3).restart();
    d.fx = d.x;
    d.fy = d.y;
  }
  function dragged(event, d) {
    d.fx = event.x;
    d.fy = event.y;
  }
  function dragended(event, d) {
    if (!event.active) sim.alphaTarget(0);
    d.fx = null;
    d.fy = null;
  }
  return d3.drag()
    .on("start", dragstarted)
    .on("drag", dragged)
    .on("end", dragended);
}
</script>
</body>
</html>
"""


def render_graph_html_d3(graph_json_path, out_path):
    """Render a NetworkX node-link graph.json as a self-contained D3.js force-directed graph.html."""
    data = json.loads(Path(graph_json_path).read_text())

    nodes = []
    for n in data["nodes"]:
        node_type = n.get("type", "unknown")
        if node_type == "text_span":
            label = n.get("text", n["id"])
        elif node_type == "attribute":
            label = n.get("value", n["id"])
        else:
            label = n.get("label", n["id"])
        nodes.append({"id": n["id"], "type": node_type, "label": label})

    raw_links = data.get("links", data.get("edges", []))
    links = [
        {"source": e["source"], "target": e["target"], "relation": e.get("relation", "")}
        for e in raw_links
    ]

    graph_data_json = json.dumps({"nodes": nodes, "links": links})
    html = _HTML_TEMPLATE.replace("__GRAPH_DATA__", graph_data_json)
    Path(out_path).write_text(html)


render_graph_html_d3("graph.json", "graph.html")

In [ ]:
from IPython.display import IFrame
IFrame("graph.html", width=920, height=620)

## (Optional) Query the graph

In [ ]:
from opengraph_image.query import query_graph
result = query_graph(graph, "What objects are in this image and how are they related?", client)
print(result.summary)

## Summary & next steps

- Read the full [opengraph-image README](https://github.com/OpenGraphAI/opengraph-ai/blob/main/opengraph-image/README.md) for the MCP server details and API reference.
- Expose this pipeline to any MCP-compatible client: `claude mcp add opengraph-image -- opengraph-image-mcp`.
- Scale from one image to a whole folder with `build_graph_from_folder(folder, output)`, which extracts every image, links equivalent entities across images, and saves one combined graph.